# 05 — Exploration des tables gold et choix des graphiques

Ce notebook explore les cinq tables gold (PostgreSQL) pour répondre aux questions de la directrice des
opérations, et choisit, graphique par graphique, la forme qui sera reprise dans le dashboard. Il se termine par un
**avant / après** : le camembert des causes de retard refait selon les principes de perception.

**pandas uniquement sur les tables gold** : elles comptent de 168 à 6 805 lignes. Les 7 millions de vols restent
dans le data lake, agrégés par Spark dans `03_gold`.

**Règles appliquées à chaque graphique (séance S04)**

| Règle | Pourquoi |
|---|---|
| Le titre énonce la **conclusion**, le sous-titre dit ce qui est mesuré | le lecteur pressé lit le titre et s'arrête là |
| Unités sur les axes, barres qui partent de 0 | une barre code par sa longueur : tronquer l'axe exagère les écarts |
| Pas de camembert, pas d'arc-en-ciel | on compare mal les angles et les aires, et des couleurs sans ordre ne portent aucun sens |
| **Une seule couleur d'accent**, et seulement si elle a un sens | ici le bleu veut toujours dire « là où agir » ; le reste est gris |
| Étiquettes directes plutôt que légende | pas d'aller-retour de l'œil entre la légende et les marques |

Les titres sont **calculés à partir des données** : relancé sur d'autres données, le notebook garde des titres
vrais.

## 1. Lecture des tables et style commun

**Ré-agréger à partir des comptes.** Dès qu'on regroupe plusieurs lignes (plusieurs mois, plusieurs compagnies),
on recalcule les taux à partir des comptes (`somme des vols à l'heure / somme des vols opérés`), jamais en
faisant la moyenne des pourcentages : un mois de 120 000 vols pèse plus qu'un mois de 6 000. Les tables de
`03_gold` contiennent ces comptes (`vols_operes`, `vols_a_l_heure`, `vols_annules`) ; si une table ne les a
pas (contrat strict), `avec_comptes()` les reconstitue à partir des pourcentages.

In [ ]:
import sys
sys.path.append("/home/jovyan/work")
from commun.donnees import lire

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter

# Palette : une seule couleur d'accent, qui veut toujours dire « là où agir » ; le reste en gris
ACCENT = "#257de0"
GRIS = "#b4b4b4"
ENCRE, ENCRE_2, GRILLE, AXE = "#191919", "#666666", "#e7e7e7", "#c2c2c2"
SEQUENTIEL = LinearSegmentedColormap.from_list(   # une seule teinte, du clair (peu) au foncé (beaucoup)
    "bleus", ["#cce3fd", "#9dc8f7", "#6aabf1", "#338ced", "#2370cd", "#1757ac", "#0e3d7e"])

plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.edgecolor": AXE, "axes.labelcolor": ENCRE_2, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlecolor": ENCRE, "axes.spines.top": False,
    "axes.spines.right": False, "grid.color": GRILLE, "grid.linewidth": 0.8,
    "xtick.color": ENCRE_2, "ytick.color": ENCRE_2, "text.color": ENCRE, "font.size": 10,
})
MOIS = ["janvier", "février", "mars", "avril", "mai", "juin", "juillet", "août", "septembre", "octobre",
        "novembre", "décembre"]
JOURS = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
EN_PCT = FuncFormatter(lambda v, _: f"{v:.0f} %")


def pct(x, d=1):
    """74.83 → « 74,8 % » (virgule décimale)."""
    return f"{x:.{d}f}".replace(".", ",") + " %"


def milliers(n):
    return f"{int(round(n)):,}".replace(",", " ")


def titrer(ax, conclusion, mesure):
    """Titre = la conclusion ; sous-titre gris = ce qui est mesuré."""
    ax.set_title(conclusion, pad=26)
    ax.text(0, 1.02, mesure, transform=ax.transAxes, color=ENCRE_2, fontsize=9.5, va="bottom")


def avec_comptes(df, vols="vols"):
    """Ajoute vols_operes / vols_a_l_heure / vols_annules s'ils manquent (approximation : les détournés,
    ≈ 0,25 % des vols, restent alors comptés comme opérés)."""
    df = df.copy()
    if "vols_operes" not in df:
        df["vols_annules"] = df[vols] * df["pct_annules"] / 100
        df["vols_operes"] = df[vols] - df["vols_annules"]
        df["vols_a_l_heure"] = df["vols_operes"] * df["pct_a_l_heure"] / 100
    return df


ponctualite = avec_comptes(lire("gold_ponctualite_mensuelle"))
causes = lire("gold_causes_retard")
aeroports = avec_comptes(lire("gold_aeroports"), vols="departs")
heure_jour = lire("gold_heure_jour")
routes = lire("gold_routes")
ponctualite["mois"] = pd.to_datetime(ponctualite["mois"])

for nom, table in [("gold_ponctualite_mensuelle", ponctualite), ("gold_causes_retard", causes),
                   ("gold_aeroports", aeroports), ("gold_heure_jour", heure_jour), ("gold_routes", routes)]:
    print(f"{nom:<27} {len(table):>5} lignes")

NATIONAL = 100 * ponctualite.vols_a_l_heure.sum() / ponctualite.vols_operes.sum()
print(f"\nPonctualité nationale 2024 : {pct(NATIONAL)} des vols opérés arrivés à l'heure "
      f"({milliers(ponctualite.vols.sum())} vols programmés)")

## 2. Contexte : la ponctualité au fil de l'année

*Les retards sont-ils saisonniers ?* (sous-question 2 du persona). Une **ligne**, car l'œil y lit une évolution
dans le temps. La ligne code par la **position** et non par la longueur : son axe n'a pas à partir de 0
(Cleveland & McGill). On garde une plage d'une trentaine de points pour ne pas grossir les écarts, et la moyenne
annuelle sert de repère, étiquetée directement.

In [ ]:
mensuel = ponctualite.groupby("mois")[["vols", "vols_operes", "vols_a_l_heure"]].sum()
mensuel["pct"] = 100 * mensuel.vols_a_l_heure / mensuel.vols_operes
creux, pic = mensuel.pct.idxmin(), mensuel.pct.idxmax()

fig, ax = plt.subplots(figsize=(10, 4.4))
ax.plot(mensuel.index, mensuel.pct, color=ACCENT, lw=2, marker="o", ms=6, mec="white", mew=1.5)
ax.axhline(NATIONAL, color=ENCRE_2, lw=0.8)
# Étiquette de la moyenne dans la marge droite, hors de la courbe
ax.annotate(f"moyenne\n{pct(NATIONAL)}", xy=(1, NATIONAL), xycoords=("axes fraction", "data"), xytext=(6, 0),
            textcoords="offset points", va="center", color=ENCRE_2, fontsize=9)
for date, place in [(creux, "top"), (pic, "bottom")]:
    decalage = -1.0 if place == "top" else 1.0
    ax.annotate(f"{MOIS[date.month - 1]} : {pct(mensuel.pct[date])}", (date, mensuel.pct[date]),
                xytext=(0, 12 * decalage), textcoords="offset points", ha="center", va=place, fontsize=9.5,
                fontweight="bold")

bas = 5 * np.floor(mensuel.pct.min() / 5) - 10
ax.set_ylim(bas, max(bas + 30, 5 * np.ceil(mensuel.pct.max() / 5) + 5))
ax.yaxis.set_major_formatter(EN_PCT)
ax.set_ylabel("Vols arrivés à l'heure (%)")
ABREGES = ["janv.", "févr.", "mars", "avr.", "mai", "juin", "juil.", "août", "sept.", "oct.", "nov.", "déc."]
ax.set_xticks(mensuel.index, [ABREGES[d.month - 1] for d in mensuel.index])
ax.grid(axis="y")
ecart = mensuel.pct[pic] - mensuel.pct[creux]
titrer(ax, f"La ponctualité varie de {ecart:.0f} points dans l'année : creux en {MOIS[creux.month - 1]}, "
           f"pic en {MOIS[pic.month - 1]}",
       "Part des vols opérés arrivés avec moins de 15 min de retard, toutes compagnies, par mois de 2024")
plt.tight_layout()
plt.show()

**Lecture.** Les creux tombent sur les mois de forte demande et de météo difficile (orages d'été, tempêtes
d'hiver) : c'est là que la directrice doit renforcer les marges. **Dans le dashboard** : même courbe, en
« Contexte », recalculée pour la compagnie et la période choisies, la moyenne de toutes les compagnies servant
de référence grise.

## 3. Les compagnies face à la moyenne nationale

*Comment se compare notre compagnie ?* (sous-question 5). Des **barres horizontales** (les noms se lisent sans
pivoter), **triées** pour que le rang se lise sans effort, **à partir de 0**. Le bleu marque les compagnies sous la
moyenne nationale ; la valeur est écrite au bout de chaque barre.

In [ ]:
cie = (ponctualite.groupby("compagnie_nom")[["vols", "vols_operes", "vols_a_l_heure"]].sum()
       .assign(pct=lambda d: 100 * d.vols_a_l_heure / d.vols_operes).sort_values("pct"))
sous = cie[cie.pct < NATIONAL]

fig, ax = plt.subplots(figsize=(9.5, 5.6))
ax.barh(cie.index, cie.pct, color=np.where(cie.pct < NATIONAL, ACCENT, GRIS), height=0.72)
for y, v in enumerate(cie.pct):
    ax.text(v + 1, y, pct(v), va="center", fontsize=9)
ax.axvline(NATIONAL, color=ENCRE, lw=0.8)
ax.text(NATIONAL, len(cie) - 0.3, f" moyenne nationale {pct(NATIONAL)}", fontsize=9, va="bottom")
ax.set_xlim(0, 100)
ax.xaxis.set_major_formatter(EN_PCT)
ax.set_xlabel("Vols arrivés à l'heure (%)")
ax.tick_params(axis="y", length=0)
titrer(ax, f"{len(sous)} compagnies sur {len(cie)} sont sous la moyenne nationale ; "
           f"{cie.index[0]} ferme la marche ({pct(cie.pct.iloc[0])})",
       "Vols opérés arrivés avec moins de 15 min de retard, 2024. En bleu : sous la moyenne nationale.")
plt.tight_layout()
plt.show()